# Faithfulness e-SNLI — Gemma3-27b-it with SAE Activation Analysis

In [1]:
import sys
import os

# Ensure src/ is on the path so `lasr` is importable.
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))

from configs import ModelConfig, InferenceConfig, PromptStyle, SAEConfig
from lasr.data import load_esnli, build_few_shot_examples, build_prompts
from lasr.inference import load_model, generate_predictions
from lasr.metrics import run_evaluation
from lasr.sae import load_sae
from lasr.activations import gather_residual_activations, encode_activations
from lasr.aggregation import top_k_features, top_k_features_per_token, reconstruction_metrics, l0_sparsity
from lasr.plotting import plot_feature_activation_heatmap, plot_per_token_topk_heatmap

# Configuration

In [2]:
model_config = ModelConfig(model_name="google/gemma-3-27b-it")
inference_config = InferenceConfig(batch_size=2, max_new_tokens=256, downsample_rate=100)
prompt_style = PromptStyle.CHAIN_OF_THOUGHT
use_few_shot = False

sae_config = SAEConfig(
    layer=40,
    width="65k",
    l0="medium",
    repo_id="google/gemma-scope-2-27b-it",
)

ESNLI_URL = "https://raw.githubusercontent.com/OanaMariaCamburu/e-SNLI/refs/heads/master/dataset/esnli_dev.csv"

print(f"Model:      {model_config.model_name}")
print(f"Device:     {model_config.device}")
print(f"Batch size: {inference_config.batch_size}")
print(f"Downsample: 1/{inference_config.downsample_rate}")
print(f"Prompt:     {prompt_style.value}")
print(f"Few-shot:   {use_few_shot}")
print(f"SAE layer:  {sae_config.layer}")
print(f"SAE width:  {sae_config.width}")
print(f"SAE l0:     {sae_config.l0}")
print(f"SAE repo:   {sae_config.repo_id}")

Model:      google/gemma-3-27b-it
Device:     cuda
Batch size: 2
Downsample: 1/100
Prompt:     chain_of_thought
Few-shot:   False
SAE layer:  40
SAE width:  65k
SAE l0:     medium
SAE repo:   google/gemma-scope-2-27b-it


# Setup — HF Token

In [3]:
import sys
from huggingface_hub import login

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import google.colab.userdata
    hf_token = google.colab.userdata.get("HFWrite")
    login(token=hf_token)
else:
    from dotenv import load_dotenv
    load_dotenv()
    hf_token = os.getenv("HF_TOKEN")

# Data — Load e-SNLI

In [4]:
esnli_df = load_esnli(ESNLI_URL)
esnli_df.head()

Downloading...
Done! 9842 rows loaded.


,pairID,gold_label,Sentence1,Sentence2,Explanation_1,Sentence1_marked_1,Sentence2_marked_1,Sentence1_Highlighted_1,Sentence2_Highlighted_1,Explanation_2,Sentence1_marked_2,Sentence2_marked_2,Sentence1_Highlighted_2,Sentence2_Highlighted_2,Explanation_3,Sentence1_marked_3,Sentence2_marked_3,Sentence1_Highlighted_3,Sentence2_Highlighted_3
0,4705552913.jpg#2r1n,neutral,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding ...,The to go packages may not be from lunch.,Two women are embracing while holding to go pa...,The sisters are hugging goodbye while holding...,{},13,"Just because two women are embracing, does not...",Two women are embracing while holding to go pa...,The *sisters* are *hugging* *goodbye* while h...,{},"1,3,4",Two women do not have to be sisters. Embracin...,Two women are embracing while holding to go pa...,*The* *sisters* are *hugging* *goodbye* while...,{},"1,0,3,4,10,11,13,12"
1,4705552913.jpg#2r1e,entailment,Two women are embracing while holding to go pa...,Two woman are holding packages.,Saying the two women are holding packages is a...,Two women are embracing while holding *to* *g...,Two woman are *holding* *packages.*,"6,7,8","3,4",Sentence 1 states that two women are holding t...,*Two* *women* are embracing while *holding* t...,Two woman are holding packages.,"0,1,5,8",{},Women can embrace while they are holding packa...,Two *women* are *embracing* while holding to ...,Two woman are *holding* *packages.*,"1,3","3,4"
2,4705552913.jpg#2r1c,contradiction,Two women are embracing while holding to go pa...,The men are fighting outside a deli.,In the first sentence there is an action of af...,Two *women* are *embracing* while holding to g...,The *men* are *fighting* outside a deli.,"1,3","1,3",Women are different than men and embracing is ...,*Two* *women* are *embracing* while holding to...,The *men* are *fighting* outside a deli.,"0,1,3","1,3",First sentence features two women and the seco...,*Two* *women* are embracing while holding to g...,The *men* are fighting outside a deli.,"0,1",1
3,2407214681.jpg#0r1e,entailment,"Two young children in blue jerseys, one with t...",Two kids in numbered jerseys wash their hands.,Young children are kids. Jerseys with number 9...,"Two *young* *children* in blue *jerseys,* one...",Two *kids* in *numbered* *jerseys* wash their...,"2,1,10,9,15,16,5","1,3,4",TWO YOUNG CHILDREN IN JERSEY WASHING THEIR HAN...,"*Two* *young* *children* in blue jerseys, one...",Two kids in numbered *jerseys* wash their *ha...,"2,26,1,0,31","4,7",Kids is a synonym for children.,"Two young *children* in blue jerseys, one wit...",Two *kids* in numbered jerseys wash their hands.,2,1
4,2407214681.jpg#0r1n,neutral,"Two young children in blue jerseys, one with t...",Two kids at a ballgame wash their hands.,Two kids in jerseys watching their hands are n...,"Two young children in blue jerseys, one with t...",Two kids *at* *a* *ballgame* wash their hands.,{},"2,3,4",Even it two children are wearing jerseys they ...,"Two young children in blue jerseys, one with t...",Two kids at a *ballgame* wash their hands.,{},4,Just because two children are in jerseys washi...,"Two young children in blue jerseys, one with t...",Two *kids* *at* *a* *ballgame* *wash* *their*...,{},"4,2,1,3,5,6,7"


# Build Prompts

In [5]:
few_shot_examples = build_few_shot_examples(esnli_df, prompt_style) if use_few_shot else None
esnli_df["prompt"] = build_prompts(esnli_df, prompt_style, few_shot=use_few_shot, few_shot_examples=few_shot_examples)

print(esnli_df["prompt"].iloc[0])

<start_of_turn>user Task: Determine the logical relationship between a Premise and a Hypothesis. 
        Options: entailment, contradiction, neutral.
        
        Rules:
        1. You MUST provide your reasoning inside <reasoning> tags.
        2. You MUST provide the final label inside <label> tags.
        3. The reasoning must come BEFORE the label.

        
        Premise: Two women are embracing while holding to go packages.
        Hypothesis: The sisters are hugging goodbye while holding to go packages after just eating lunch.
        
<end_of_turn>model


# Load Model + SAE

In [6]:
model, tokenizer = load_model(model_config)

Loading weights:   0%|          | 0/1247 [00:00<?, ?it/s]

In [7]:
sae = load_sae(sae_config)

## Gather activations for generated tokens

In [ ]:
import torch

# Pick the first sample prompt
sample_prompt = esnli_df["prompt"].iloc[0]
prompt_ids = tokenizer.encode(sample_prompt, return_tensors="pt", add_special_tokens=True).to(model_config.device)
full_ids = model.generate(input_ids=prompt_ids, max_new_tokens=inference_config.max_new_tokens)
full_response = tokenizer.decode(full_ids[0], skip_special_tokens=True)
print(f"Full response: {full_response}")
print("-" * 25)
prompt_len = prompt_ids.shape[1]
print(f"Prompt tokens: {prompt_len}")
gen_len = full_ids.shape[1] - prompt_len
print(f"Generated tokens: {gen_len}")
print(f"Full sequence tokens: {full_ids.shape[1]}")

# Run a forward pass on the full sequence and hook the SAE target layer
residual_acts = gather_residual_activations(model, sae_config.layer, full_ids)
print(f"Residual activations shape (full): {residual_acts.shape}")

# Slice to keep only the generated-token activations (for reconstruction metrics)
gen_acts = residual_acts[:, prompt_len:, :]
print(f"Generated-only activations shape: {gen_acts.shape}")

# Encode *full sequence* through the SAE
sae_acts_full, _ = encode_activations(sae, residual_acts)
print(f"SAE feature activations shape (full): {sae_acts_full.shape}")

# Denoise SAE activations (TF-IDF weighting)
from lasr.denoising import DenoisingConfig, denoise

denoising_config = DenoisingConfig()  # defaults to continuous_tfidf
sae_acts_full = denoise(sae_acts_full, denoising_config)
print(f"Denoised SAE activations shape: {sae_acts_full.shape}")

# Derive generation-only slice for reconstruction metrics
sae_acts_gen = sae_acts_full[:, prompt_len:, :]
_, reconstruction = encode_activations(sae, gen_acts)
print(f"SAE feature activations shape (gen): {sae_acts_gen.shape}")
print(f"Reconstruction shape: {reconstruction.shape}")

# Build token strings for the full sequence
all_tokens = tokenizer.convert_ids_to_tokens(full_ids[0])
print(f"All tokens: {len(all_tokens)}")


KeyboardInterrupt: 

In [9]:
# Reconstruction quality (generation-only slice)
metrics = reconstruction_metrics(reconstruction, gen_acts)
print(f"MSE: {metrics['mse']:.6f}")
print(f"FVU: {metrics['fvu']:.6f}")

MSE: 14490.027344
FVU: 0.021197


In [10]:
# L0 sparsity (generation-only slice)
l0 = l0_sparsity(sae_acts_gen)
print(f"Per-token L0: {l0}")
print(f"Average L0: {l0.float().mean():.1f}")

Per-token L0: tensor([[ 57,  61,  60,  79,  62,  54,  53,  85,  86,  88,  80,  79,  70,  61,
          57,  54,  62,  67,  85,  45,  43,  64,  52,  71,  47, 103,  81,  68,
          72,  59,  92,  62,  73,  60,  78,  53,  57,  68,  70,  46,  61,  56,
          67,  63,  74,  75,  65,  59,  49,  71,  50,  92,  48,  88,  70,  59,
          79,  86,  74,  72,  61,  56,  55,  68,  65,  55,  69,  54,  55,  55,
          83, 102,  93,  49,  88,  70,  49,  50,  92,  58,  67,  61,  62,  68,
          39,  33,  67,  74,  78,  63,  71,  60,  63,  54,  46,  46,  55,  67,
          73,  42,  77,  85,  78,  98,  62,  66,  45,  82,  81,  59,  72,  82,
          60,  84,  76,  51,  62,  55,  45,  61,  42,  37,  51,  75,  42,  53,
          49,  71,  51,  53,  35,  51,  38,  38]], device='cuda:0')
Average L0: 64.2


In [11]:
# Top-K features per token (generation-only slice)
from neuronpedia_client import build_sae_id

TOP_K = 10
per_token_vals, per_token_idxs = top_k_features_per_token(sae_acts_gen, k=TOP_K)
print(f"Per-token top-{TOP_K} values shape: {per_token_vals.shape}")
print(f"Per-token top-{TOP_K} indices shape: {per_token_idxs.shape}")

# Collect all unique feature indices across every token
unique_features = sorted(set(per_token_idxs.cpu().numpy().ravel().tolist()))
print(f"Unique features across all tokens: {len(unique_features)}")

np_model_id = "gemma-3-27b-it"
np_sae_id = build_sae_id(sae_config)

# Show top-K for first 3 tokens (labels will come from Feature objects later)
gen_token_ids = full_ids[0, prompt_len:]
tokens = tokenizer.convert_ids_to_tokens(gen_token_ids)
print(f"\nTop {TOP_K} SAE features per token (first 3 tokens shown):")
for t in range(min(3, len(tokens))):
    print(f"\n  Token '{tokens[t]}':")
    for r in range(TOP_K):
        idx = int(per_token_idxs[t, r])
        val = float(per_token_vals[t, r])
        print(f"    Rank {r+1}  feature {idx:>5d}  activation = {val:.4f}")

Per-token top-10 values shape: torch.Size([134, 10])
Per-token top-10 indices shape: torch.Size([134, 10])
Unique features across all tokens: 606

Top 10 SAE features per token (first 3 tokens shown):

  Token '
':
    Rank 1  feature  1284  activation = 3112.3679
    Rank 2  feature  1663  activation = 2730.0083
    Rank 3  feature 17135  activation = 1817.4219
    Rank 4  feature 46045  activation = 1588.8695
    Rank 5  feature  2108  activation = 1355.4926
    Rank 6  feature 63049  activation = 1353.8401
    Rank 7  feature   984  activation = 1239.0613
    Rank 8  feature 31241  activation = 1213.8741
    Rank 9  feature 19903  activation = 1162.1936
    Rank 10  feature  2208  activation = 1139.6909

  Token '<':
    Rank 1  feature  3448  activation = 2583.6909
    Rank 2  feature  6720  activation = 2574.2002
    Rank 3  feature  1284  activation = 2095.7920
    Rank 4  feature  1411  activation = 1669.0562
    Rank 5  feature  2199  activation = 1575.6230
    Rank 6  feature 

## Activation Heatmap

In [12]:
# Create Feature objects (full sequence activations)
from lasr.feature import create_features

features = create_features(sae_acts_full, unique_features, all_tokens)

# Fetch Neuronpedia details
for f in features:
    f.fetch_details(np_model_id, np_sae_id)

feature_map = {f.feature_idx: f for f in features}
print(f"Created {len(features)} Feature objects")
print(f"Example: {feature_map[unique_features[0]]!r}")

# Build labels dict for the heatmap
labels = {f.feature_idx: f.label for f in features}

fig = plot_per_token_topk_heatmap(
    per_token_vals,
    per_token_idxs,
    tokens=tokens,
    labels=labels,
    title="Gemma3-27b-it Per-Token Top-K SAE Feature Activations (Layer 40)",
)
fig.show()

Created 606 Feature objects
Example: Feature(idx=8, label='evaluating complexity and worth', max_act=11244.54, n_tokens=255)


## Inspect Feature — Neuronpedia Embed + Token Highlighting

In [13]:
# --- Inspect features using the Feature class ---
example_feature = int(per_token_idxs[0, 0])
print(f"Inspecting feature {example_feature} (top feature for first generated token)")
print(f"repr: {feature_map[example_feature]!r}")
print(f"frac_nonzero: {feature_map[example_feature].frac_nonzero}")
print(f"top_tokens: {feature_map[example_feature].top_tokens()}")
print()

# Full sequence view
feature_map[example_feature].inspect(prompt_length=prompt_len)

# Generation-only view
feature_map[example_feature].inspect(token_range="generation", prompt_length=prompt_len)

Inspecting feature 1284 (top feature for first generated token)
repr: Feature(idx=1284, label='model responses', max_act=26158.89, n_tokens=255)
frac_nonzero: 0.004693163724031576
top_tokens: [('model', -26158.88671875), ('\n\n', -21246.982421875), ('\n', -16775.474609375), ('\n', -16006.359375), ('\n', -14061.390625)]

